# OnKith research figures

Every table and plot for the report, built from evidence already in the
repository. This notebook **trains nothing, evaluates nothing and downloads
nothing** - it reads saved metrics and draws them.

Run All. Each section either produces its figure or says exactly which artifact
is missing; a missing input never stops the notebook.

**Outputs**

```text
research_results/
├── figures/      PNG (300 dpi) and PDF for every plot
├── tables/       the numbers behind each figure, as CSV
└── summaries/    figure index and a written overview
```

**Deployment figures.** Sections 9 and 10 read whatever
`Full_Pipeline_CPU_Benchmark.py` and `Full_Pipeline_Audio_Benchmark.py` have
left under `research_results/cpu_benchmarks/`. Section 9 covers the 2,000-row
text benchmark; section 10 covers the end-to-end audio benchmark, from a WAV
file to sanitized text. Before either has been run there is nothing to draw and
the section says so. Once the Raspberry Pi results are committed and pulled,
rerunning this notebook adds the device-comparison and parity figures with no
edits.

**Honesty rule.** A metric is only plotted beside another metric when the two
were measured the same way. Where a historical model has no comparable number,
this notebook says "not recorded" rather than inventing one or quietly
substituting a different metric.

## 0. Setup

In [ ]:
import csv
import json
import math
import statistics
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # figures are files, not windows; keeps Run All headless-safe
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

REPO_ROOT = Path.cwd()
while not (REPO_ROOT / "FullPipeline_V3").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / "FullPipeline_V3").is_dir():
    raise RuntimeError("Run this notebook from inside the OnKith repository.")

RESULTS = REPO_ROOT / "research_results"
FIGURES = RESULTS / "figures"
TABLES = RESULTS / "tables"
SUMMARIES = RESULTS / "summaries"
CPU_BENCHMARKS = RESULTS / "cpu_benchmarks"
for directory in (FIGURES, TABLES, SUMMARIES, CPU_BENCHMARKS):
    directory.mkdir(parents=True, exist_ok=True)

# --- publication styling -----------------------------------------------------
# Categorical hues are assigned in a fixed order and never cycled: blue is always
# the current deployment model, orange the model it replaced, aqua the third
# series where one exists.
SERIES = {"primary": "#2a78d6", "secondary": "#eb6834", "tertiary": "#1baf7a"}
INK = "#0b0b0b"
MUTED = "#52514e"
GRID = "#d9d8d4"

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "font.size": 10.5,
    "axes.titlesize": 12.5,
    "axes.labelsize": 10.5,
    "axes.titleweight": "bold",
    "axes.edgecolor": GRID,
    "axes.labelcolor": INK,
    "axes.titlecolor": INK,
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": GRID,
    "grid.linewidth": 0.7,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "xtick.labelsize": 9.5,
    "ytick.labelsize": 9.5,
    "legend.frameon": False,
    "legend.fontsize": 9.5,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "lines.linewidth": 2.0,
    "lines.markersize": 6,
})

GENERATED: list[dict] = []      # every figure and table this run produced
UNAVAILABLE: list[str] = []     # every input that was missing, with the reason


def tidy(axis, *, x_grid: bool = False) -> None:
    """Recessive frame: no top/right spines, grid on the value axis only."""

    axis.spines["top"].set_visible(False)
    axis.spines["right"].set_visible(False)
    axis.grid(axis="x" if x_grid else "y")
    axis.set_axisbelow(True)


def save(figure, name: str, caption: str) -> None:
    """PNG for the report, PDF for the vector version, both at print quality."""

    png, pdf = FIGURES / f"{name}.png", FIGURES / f"{name}.pdf"
    figure.savefig(png)
    figure.savefig(pdf)
    plt.close(figure)
    GENERATED.append({"kind": "figure", "name": name,
                      "files": [png.name, f"{pdf.name}"], "caption": caption})
    print(f"  figure  {png.relative_to(REPO_ROOT).as_posix()}  (+ .pdf)")


def save_table(rows, fieldnames, name: str, caption: str) -> None:
    if not rows:
        return
    path = TABLES / f"{name}.csv"
    with path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=fieldnames, lineterminator="\n")
        writer.writeheader()
        writer.writerows(rows)
    GENERATED.append({"kind": "table", "name": name, "files": [path.name],
                      "caption": caption, "rows": len(rows)})
    print(f"  table   {path.relative_to(REPO_ROOT).as_posix()}  ({len(rows)} rows)")


def load_json(path: Path, label: str):
    if not Path(path).is_file():
        UNAVAILABLE.append(f"{label}: {Path(path).relative_to(REPO_ROOT).as_posix()} not found")
        return None
    try:
        return json.loads(Path(path).read_text(encoding="utf-8"))
    except json.JSONDecodeError as exc:
        UNAVAILABLE.append(f"{label}: unreadable ({exc})")
        return None


print("repository :", REPO_ROOT)
print("figures    :", FIGURES.relative_to(REPO_ROOT).as_posix())
print("tables     :", TABLES.relative_to(REPO_ROOT).as_posix())

## 1. Discover the evidence

Everything this notebook can draw, and whether it is present.

In [ ]:
MODEL_V2 = REPO_ROOT / "model" / "DeBERTa-v3-xsmall"
TINYBERT_V1 = REPO_ROOT / "model" / "TinyBERT-4" / "artifacts" / "tinybert_pii"
METRICS = MODEL_V2 / "artifacts" / "model_v2" / "metrics"
EVALUATION = REPO_ROOT / "model" / "privacy_evaluation"

DEBERTA_FP32 = load_json(METRICS / "evaluation_production_path_fp32_onnx.json", "DeBERTa FP32 ONNX")
DEBERTA_INT8 = load_json(METRICS / "evaluation_production_path_int8_onnx.json", "DeBERTa INT8 ONNX")
ACCEPTED = load_json(METRICS / "accepted_int8_candidate.json", "accepted INT8 candidate")
OOD_FINAL_DEBERTA = load_json(METRICS / "evaluation_ood_final_int8.json", "DeBERTa frozen OOD final")
DEPLOY_MANIFEST = load_json(
    REPO_ROOT / "FullPipeline_V3" / "models" / "deberta_pii" / "model_manifest.json",
    "deployment manifest",
)

TINYBERT_IID = load_json(EVALUATION / "baselines" / "tinybert_iid_baseline.json", "TinyBERT IID baseline")
TINYBERT_OOD_DEV = load_json(EVALUATION / "results" / "tinybert_dev_summary.json", "TinyBERT OOD dev")
TINYBERT_OOD_FINAL = load_json(EVALUATION / "results" / "tinybert_final_summary.json", "TinyBERT OOD final")

BILSTM = load_json(REPO_ROOT / "model" / "artifacts" / "final_eval_metrics.json", "BiLSTM baseline")
PREFLIGHT = load_json(
    MODEL_V2 / "artifacts" / "preflight" / "metrics" / "final_benchmark.json",
    "workstation preflight benchmark",
)

VIEW_ORDER = ["test", "val_original", "val_supplementary", "combined_validation", "ood_dev"]
VIEW_LABELS = {
    "test": "Test",
    "val_original": "Validation\n(original)",
    "val_supplementary": "Validation\n(supplementary)",
    "combined_validation": "Combined\nvalidation",
    "ood_dev": "OOD dev",
    "ood_final": "OOD final\n(frozen)",
}
# Short forms for the axes that carry six groups; the descriptive names above
# stay available for captions and prose.
VIEW_TICKS = {
    "test": "Test",
    "val_original": "Val\n(orig)",
    "val_supplementary": "Val\n(supp)",
    "combined_validation": "Combined\nval",
    "ood_dev": "OOD\ndev",
    "ood_final": "OOD final\n(frozen)",
}

sources = [
    ("DeBERTa INT8, five evaluation views", DEBERTA_INT8),
    ("DeBERTa FP32, five evaluation views", DEBERTA_FP32),
    ("DeBERTa frozen OOD final (one-shot)", OOD_FINAL_DEBERTA),
    ("Accepted INT8 candidate / quality gate", ACCEPTED),
    ("TinyBERT IID baseline (historical)", TINYBERT_IID),
    ("TinyBERT OOD dev", TINYBERT_OOD_DEV),
    ("TinyBERT OOD final", TINYBERT_OOD_FINAL),
    ("BiLSTM baseline", BILSTM),
    ("Workstation latency preflight", PREFLIGHT),
]
print(f"{'source':44} available")
print("-" * 56)
for label, payload in sources:
    print(f"{label:44} {'yes' if payload else 'NO'}")

def benchmark_kind(directory: Path) -> str:
    """Text and audio runs share one results root; the config says which is which."""

    try:
        config = json.loads((directory / "benchmark_config.json").read_text(encoding="utf-8"))
    except (OSError, json.JSONDecodeError):
        return "unreadable"
    # Runs recorded before the audio benchmark existed carry no marker, and they
    # are all text runs.
    return str(config.get("benchmark", "cpu_text"))


discovered = sorted(
    directory for directory in CPU_BENCHMARKS.glob("*")
    if directory.is_dir() and (directory / "benchmark_config.json").is_file()
)
benchmark_runs = [d for d in discovered if benchmark_kind(d) == "cpu_text"]
audio_benchmark_runs = [d for d in discovered if benchmark_kind(d) == "audio_pipeline"]

print(f"\nCPU text benchmark runs found : {len(benchmark_runs)}")
for directory in benchmark_runs:
    print(f"  {directory.name}")
print(f"audio pipeline benchmark runs : {len(audio_benchmark_runs)}")
for directory in audio_benchmark_runs:
    print(f"  {directory.name}")

## 2. Model progression

Three generations of privacy model. Only numbers measured the same way are put
on the same axis:

* **BiLSTM** was scored with token-level *binary* accuracy on its own validation
  split. There is no span-level evaluation for it, so it has no bar here. Its
  recorded number is in the table, flagged as not comparable.
* **TinyBERT-4 INT8** and **DeBERTa-v3-xsmall INT8** were scored on the *same*
  40,909-row held-out AI4Privacy English validation split, through the same
  metric definitions, in `deberta_vs_tinybert_int8_validation.ipynb`. Those two
  are directly comparable and are what the figure shows.

In [ ]:
progression_rows = []
head_to_head = (DEPLOY_MANIFEST or {}).get("held_out_validation")

if BILSTM:
    progression_rows.append({
        "model": "BiLSTM baseline",
        "generation": "v0",
        "precision": "fp32",
        "metric": "token-level binary F1",
        "value": BILSTM["val_f1"],
        "precision_value": BILSTM["val_precision"],
        "recall_value": BILSTM["val_recall"],
        "evaluation": "own validation split",
        "comparable_to_span_metrics": False,
    })

if head_to_head:
    for key, name, generation in (
        ("tinybert_int8_model_v1", "TinyBERT-4 INT8", "v1"),
        ("deberta_int8", "DeBERTa-v3-xsmall INT8", "v2"),
    ):
        entry = head_to_head[key]
        progression_rows.append({
            "model": name,
            "generation": generation,
            "precision": "int8",
            "metric": "typed entity F1",
            "value": entry["typed_f1"],
            "precision_value": None,
            "recall_value": None,
            "evaluation": f"shared held-out validation, {head_to_head['rows']:,} rows",
            "comparable_to_span_metrics": True,
            "private_character_leakage": entry["private_character_leakage"],
            "non_private_overmasking_rate": entry["non_private_overmasking_rate"],
        })

if head_to_head:
    comparable = [row for row in progression_rows if row["comparable_to_span_metrics"]]
    names = [row["model"] for row in comparable]
    f1 = [row["value"] for row in comparable]
    leakage = [row["private_character_leakage"] for row in comparable]
    overmask = [row["non_private_overmasking_rate"] for row in comparable]

    figure, (left, right) = plt.subplots(1, 2, figsize=(10.5, 4.2))
    colours = [SERIES["secondary"], SERIES["primary"]]

    # Bars start at zero. The two models are genuinely close on this split, and a
    # cropped axis that turned 0.0086 F1 into a towering difference would be a lie;
    # the gap that matters is in the error rates on the right.
    bars = left.bar(names, f1, color=colours, width=0.55)
    for bar, value in zip(bars, f1):
        left.annotate(f"{value:.4f}", (bar.get_x() + bar.get_width() / 2, value),
                      ha="center", va="bottom", xytext=(0, 4),
                      textcoords="offset points", fontsize=10, color=INK)
    left.set_ylim(0, 1.08)
    left.set_ylabel("Typed entity F1")
    left.set_title(f"Exact-span, correct-type F1  (+{f1[1] - f1[0]:.4f})")
    tidy(left)

    width = 0.35
    positions = range(len(names))
    error_bars_leak = right.bar([p - width / 2 for p in positions], leakage,
                                width=width, color=colours[0], label="Private character leakage")
    error_bars_over = right.bar([p + width / 2 for p in positions], overmask,
                                width=width, color=colours[1], label="Non-private overmasking")
    for group, values in ((error_bars_leak, leakage), (error_bars_over, overmask)):
        for bar, value in zip(group, values):
            right.annotate(f"{value:.4f}", (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=8.5, color=MUTED)
    right.set_xticks(list(positions), names)
    right.set_ylabel("Rate (lower is better)")
    right.set_title("Character-level error rates")
    right.set_ylim(0, max(leakage + overmask) * 1.6)
    right.legend(loc="upper center", ncol=2)
    tidy(right)

    figure.suptitle(
        "Privacy model generations on the shared held-out validation split "
        f"({head_to_head['rows']:,} rows)",
        fontsize=13, y=1.02,
    )
    figure.text(
        0.5, -0.06,
        "The earlier BiLSTM baseline has no span-level evaluation on record "
        f"(token-level binary F1 {BILSTM['val_f1']:.4f} on its own split), so it is "
        "listed in the table rather than plotted here."
        if BILSTM else "",
        ha="center", fontsize=9, color=MUTED,
    )
    save(figure, "model_progression",
         "Typed entity F1 and character-level error rates for TinyBERT INT8 and "
         "DeBERTa INT8 on the same held-out validation split.")
else:
    print("skipped: no shared head-to-head validation record")

save_table(
    progression_rows,
    ["model", "generation", "precision", "metric", "value", "precision_value",
     "recall_value", "evaluation", "comparable_to_span_metrics",
     "private_character_leakage", "non_private_overmasking_rate"],
    "model_progression",
    "Model generations with the metric each one actually has on record.",
)

## 3. Quantization: FP32 against INT8

Both runtimes were decoded by the same production decoder, so the difference is
quantization and nothing else. The dashed line is the release gate: a view fails
if it loses more than 0.02 typed entity F1.

In [ ]:
GATE_LIMIT = 0.02
quantization_rows = []

if DEBERTA_FP32 and DEBERTA_INT8:
    fp32 = DEBERTA_FP32["summary"]
    int8 = DEBERTA_INT8["summary"]
    views = [view for view in VIEW_ORDER if view in fp32 and view in int8]

    for view in views:
        quantization_rows.append({
            "view": view,
            "rows": int8[view]["rows"],
            "fp32_typed_f1": fp32[view]["typed_entity_f1"],
            "int8_typed_f1": int8[view]["typed_entity_f1"],
            "typed_f1_drop": fp32[view]["typed_entity_f1"] - int8[view]["typed_entity_f1"],
            "fp32_leakage": fp32[view]["private_character_leakage"],
            "int8_leakage": int8[view]["private_character_leakage"],
            "leakage_rise": int8[view]["private_character_leakage"]
            - fp32[view]["private_character_leakage"],
            "fp32_character_recall": fp32[view]["private_character_recall"],
            "int8_character_recall": int8[view]["private_character_recall"],
        })

    figure, (top, bottom) = plt.subplots(
        2, 1, figsize=(9.5, 7.0), gridspec_kw={"height_ratios": [2.1, 1]}
    )
    positions = range(len(views))
    width = 0.38
    fp32_values = [fp32[view]["typed_entity_f1"] for view in views]
    int8_values = [int8[view]["typed_entity_f1"] for view in views]

    fp32_bars = top.bar([p - width / 2 for p in positions], fp32_values, width=width,
                        color=SERIES["tertiary"], label="FP32 ONNX")
    int8_bars = top.bar([p + width / 2 for p in positions], int8_values, width=width,
                        color=SERIES["primary"], label="INT8 ONNX (deployed)")
    for group, values in ((fp32_bars, fp32_values), (int8_bars, int8_values)):
        for bar, value in zip(group, values):
            top.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                         ha="center", va="bottom", xytext=(0, 3),
                         textcoords="offset points", fontsize=8.5, color=INK)
    top.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    top.set_ylabel("Typed entity F1")
    top.set_ylim(0, 1.30)
    top.set_title("Typed entity F1 by evaluation view")
    top.legend(loc="upper left", ncol=2)
    tidy(top)

    drops = [row["typed_f1_drop"] for row in quantization_rows]
    drop_bars = bottom.bar(list(positions), drops, width=0.5, color=SERIES["secondary"])
    for bar, value in zip(drop_bars, drops):
        bottom.annotate(f"{value:+.4f}", (bar.get_x() + bar.get_width() / 2, value),
                        ha="center", va="bottom", xytext=(0, 3),
                        textcoords="offset points", fontsize=9, color=INK)
    bottom.axhline(GATE_LIMIT, linestyle="--", linewidth=1.4, color=MUTED)
    bottom.annotate(f"release gate {GATE_LIMIT}", (-0.45, GATE_LIMIT),
                    ha="left", va="bottom", xytext=(0, 4), textcoords="offset points",
                    fontsize=9, color=MUTED)
    bottom.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    bottom.set_ylabel("F1 lost to INT8")
    bottom.set_ylim(0, max(GATE_LIMIT, max(drops)) * 1.75)
    bottom.set_title("Cost of quantization, against the acceptance limit")
    tidy(bottom)

    figure.suptitle("DeBERTa-v3-xsmall: FP32 against INT8 through one decoder",
                    fontsize=13, y=0.98)
    figure.tight_layout(rect=(0, 0, 1, 0.95))
    save(figure, "deberta_fp32_vs_int8",
         "FP32 and INT8 typed entity F1 on every evaluation view, with the F1 lost "
         "to quantization against the 0.02 release gate.")

    print(f"{'view':22} {'FP32':>8} {'INT8':>8} {'drop':>9} {'gate':>6}")
    for row in quantization_rows:
        print(f"{row['view']:22} {row['fp32_typed_f1']:>8.4f} {row['int8_typed_f1']:>8.4f} "
              f"{row['typed_f1_drop']:>+9.4f} {'PASS' if row['typed_f1_drop'] < GATE_LIMIT else 'FAIL':>6}")
else:
    print("skipped: the FP32 or INT8 production-path evaluation is missing")

save_table(
    quantization_rows,
    ["view", "rows", "fp32_typed_f1", "int8_typed_f1", "typed_f1_drop",
     "fp32_leakage", "int8_leakage", "leakage_rise",
     "fp32_character_recall", "int8_character_recall"],
    "deberta_quantization",
    "FP32 against INT8 on every view, with the deltas the release gate reads.",
)

## 4. Where the deployed model is strong and where it is not

The same INT8 artifact across every view it was measured on, including the
one-shot frozen OOD final benchmark. The in-distribution views and the
out-of-distribution views are deliberately drawn together: the gap between them
is the honest statement of what this model does on speech it has never seen.

In [ ]:
split_rows = []

if DEBERTA_INT8:
    summary = dict(DEBERTA_INT8["summary"])
    if OOD_FINAL_DEBERTA:
        headline = OOD_FINAL_DEBERTA["headline"]
        metrics = OOD_FINAL_DEBERTA["metrics"]
        summary["ood_final"] = {
            "typed_entity_f1": headline["typed_entity_f1"],
            "typed_entity_precision": metrics["typed_entity"]["precision"],
            "typed_entity_recall": metrics["typed_entity"]["recall"],
            "binary_entity_f1": headline["binary_entity_f1"],
            "private_character_recall": headline["private_character_recall"],
            "private_character_leakage": headline["private_character_leakage"],
            "non_private_overmasking_rate": headline["non_private_overmasking_rate"],
            "hard_negative_false_positive_rate": headline["hard_negative_false_positive_rate"],
            "rows": OOD_FINAL_DEBERTA["fixture"]["case_count"],
        }

    views = [view for view in VIEW_ORDER + ["ood_final"] if view in summary]
    for view in views:
        entry = summary[view]
        split_rows.append({
            "view": view,
            "rows": entry["rows"],
            "typed_precision": entry["typed_entity_precision"],
            "typed_recall": entry["typed_entity_recall"],
            "typed_f1": entry["typed_entity_f1"],
            "binary_f1": entry["binary_entity_f1"],
            "private_character_recall": entry["private_character_recall"],
            "private_character_leakage": entry["private_character_leakage"],
            "non_private_overmasking_rate": entry["non_private_overmasking_rate"],
            "hard_negative_false_positive_rate": entry["hard_negative_false_positive_rate"],
        })

    figure, (left, right) = plt.subplots(1, 2, figsize=(13.5, 4.8))
    positions = range(len(views))
    width = 0.27
    triples = (
        ("Precision", [row["typed_precision"] for row in split_rows], SERIES["tertiary"]),
        ("Recall", [row["typed_recall"] for row in split_rows], SERIES["secondary"]),
        ("F1", [row["typed_f1"] for row in split_rows], SERIES["primary"]),
    )
    for index, (name, values, colour) in enumerate(triples):
        offset = (index - 1) * width
        bars = left.bar([p + offset for p in positions], values, width=width,
                        color=colour, label=name)
        if name == "F1":
            for bar, value in zip(bars, values):
                left.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=8.5, color=INK)
    left.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    left.set_ylim(0, 1.32)
    left.set_ylabel("Typed entity score")
    left.set_title("Exact-span, correct-type performance")
    left.legend(loc="upper left", ncol=3)
    tidy(left)

    recall = [row["private_character_recall"] for row in split_rows]
    bars = right.bar(list(positions), recall, width=0.55, color=SERIES["primary"])
    for bar, value in zip(bars, recall):
        right.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9, color=INK)
    right.set_xticks(list(positions), [VIEW_TICKS[view] for view in views])
    right.set_ylim(0, 1.15)
    right.set_ylabel("Private characters covered")
    right.set_title("Character recall - the privacy-relevant measure")
    right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    tidy(right)

    figure.suptitle("Deployed DeBERTa-v3-xsmall INT8 across every evaluation view",
                    fontsize=13, y=1.01)
    save(figure, "deberta_split_performance",
         "Typed entity precision/recall/F1 and private-character recall for the "
         "deployed INT8 model on each evaluation view.")

    for row in split_rows:
        print(f"{row['view']:22} rows {row['rows']:>7,}  typed F1 {row['typed_f1']:.4f}  "
              f"char recall {row['private_character_recall']:.4f}  "
              f"leakage {row['private_character_leakage']:.4f}")
else:
    print("skipped: the INT8 production-path evaluation is missing")

save_table(
    split_rows,
    ["view", "rows", "typed_precision", "typed_recall", "typed_f1", "binary_f1",
     "private_character_recall", "private_character_leakage",
     "non_private_overmasking_rate", "hard_negative_false_positive_rate"],
    "deberta_split_performance",
    "Every metric of the deployed INT8 model on every evaluation view.",
)

## 5. Per-entity performance

Entity types, not BIO labels: 31 entity rows are readable, 63 tag rows are not,
and the entity is the unit the mask policy is written in. Measured on the
combined validation view, the same one checkpoint selection used.

Bars are ordered worst to best, and each is annotated with its gold support, so
a low F1 on three spans is not mistaken for a low F1 on three thousand.

In [ ]:
per_entity_rows = []

if DEBERTA_INT8 and "combined_validation" in DEBERTA_INT8["views"]:
    per_label = DEBERTA_INT8["views"]["combined_validation"]["per_label"]
    per_entity_rows = sorted(
        (
            {
                "entity": entity,
                "precision": entry["precision"],
                "recall": entry["recall"],
                "f1": entry["f1"],
                "support": entry["support"],
                "predicted": entry["predicted"],
                "true_positives": entry["true_positives"],
            }
            for entity, entry in per_label.items()
        ),
        key=lambda row: row["f1"],
    )

    height = max(5.0, 0.30 * len(per_entity_rows) + 1.4)
    figure, axis = plt.subplots(figsize=(9.0, height))
    labels = [row["entity"] for row in per_entity_rows]
    values = [row["f1"] for row in per_entity_rows]
    positions = range(len(labels))
    bars = axis.barh(list(positions), values, height=0.62, color=SERIES["primary"])
    for bar, row in zip(bars, per_entity_rows):
        axis.annotate(
            f"{row['f1']:.3f}   n={row['support']:,}",
            (row["f1"], bar.get_y() + bar.get_height() / 2),
            va="center", ha="left", xytext=(5, 0), textcoords="offset points",
            fontsize=8.5, color=MUTED,
        )
    axis.set_yticks(list(positions), labels)
    axis.set_xlim(0, 1.18)
    axis.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    axis.set_xlabel("Typed entity F1 (exact span, correct type)")
    axis.set_title(
        "Per-entity performance, deployed INT8 model\n"
        f"combined validation, {DEBERTA_INT8['summary']['combined_validation']['rows']:,} rows",
        loc="left",
    )
    tidy(axis, x_grid=True)
    save(figure, "per_entity_f1",
         "Typed entity F1 per entity type on the combined validation view, "
         "annotated with gold support.")

    weakest = per_entity_rows[:5]
    print("weakest entity types:")
    for row in weakest:
        print(f"  {row['entity']:20} F1 {row['f1']:.4f}  support {row['support']:,}")
else:
    print("skipped: no per-label breakdown available")

save_table(
    per_entity_rows,
    ["entity", "precision", "recall", "f1", "support", "predicted", "true_positives"],
    "per_entity_f1",
    "Per-entity precision, recall, F1 and support on the combined validation view.",
)

## 6. Out-of-distribution: the two deployment models on the same fixtures

The strongest comparison in the project. Both models were scored on the two
purpose-built OOD fixtures - 360 development cases and 360 frozen final cases,
byte-identical files, the same metric code, both in their model-only form. The
frozen set was spent exactly once per model.

This is where a model trained on generated corpora meets speech-shaped input it
has never seen, and it separates the two generations far more sharply than the
in-distribution split does.

In [ ]:
def tinybert_ood(summary, stage: str = "int8_model"):
    if not summary or stage not in summary.get("stages", {}):
        return None
    overall = summary["stages"][stage]["overall"]
    return {
        "typed_f1": overall["typed_entity"]["f1"],
        "typed_precision": overall["typed_entity"]["precision"],
        "typed_recall": overall["typed_entity"]["recall"],
        "character_recall": overall["privacy_character"]["private_character_recall"],
        "character_leakage": overall["privacy_character"]["private_character_leakage_rate"],
        "overmasking": overall["privacy_character"]["non_private_character_overmasking_rate"],
        "hard_negative_fp": overall["case_level"]["hard_negative_false_positive_case_rate"],
        "rows": overall["case_count"],
    }


def deberta_ood(view: str):
    if view == "ood_final":
        if not OOD_FINAL_DEBERTA:
            return None
        metrics, headline = OOD_FINAL_DEBERTA["metrics"], OOD_FINAL_DEBERTA["headline"]
        return {
            "typed_f1": headline["typed_entity_f1"],
            "typed_precision": metrics["typed_entity"]["precision"],
            "typed_recall": metrics["typed_entity"]["recall"],
            "character_recall": headline["private_character_recall"],
            "character_leakage": headline["private_character_leakage"],
            "overmasking": headline["non_private_overmasking_rate"],
            "hard_negative_fp": headline["hard_negative_false_positive_rate"],
            "rows": OOD_FINAL_DEBERTA["fixture"]["case_count"],
        }
    if not DEBERTA_INT8 or view not in DEBERTA_INT8["summary"]:
        return None
    entry = DEBERTA_INT8["summary"][view]
    return {
        "typed_f1": entry["typed_entity_f1"],
        "typed_precision": entry["typed_entity_precision"],
        "typed_recall": entry["typed_entity_recall"],
        "character_recall": entry["private_character_recall"],
        "character_leakage": entry["private_character_leakage"],
        "overmasking": entry["non_private_overmasking_rate"],
        "hard_negative_fp": entry["hard_negative_false_positive_rate"],
        "rows": entry["rows"],
    }


ood_rows = []
pairs = [
    ("OOD dev", tinybert_ood(TINYBERT_OOD_DEV), deberta_ood("ood_dev")),
    ("OOD final (frozen)", tinybert_ood(TINYBERT_OOD_FINAL), deberta_ood("ood_final")),
]
usable = [(name, old, new) for name, old, new in pairs if old and new]

for name, old, new in usable:
    for model, entry in (("TinyBERT-4 INT8", old), ("DeBERTa-v3-xsmall INT8", new)):
        ood_rows.append({"fixture": name, "model": model, **entry})

if usable:
    figure, axes = plt.subplots(1, 3, figsize=(13.0, 4.4))
    panels = (
        ("typed_f1", "Typed entity F1", "higher is better"),
        ("character_recall", "Private character recall", "higher is better"),
        ("hard_negative_fp", "Hard-negative false-positive rate", "lower is better"),
    )
    names = [name for name, _, _ in usable]
    positions = range(len(names))
    width = 0.36

    for axis, (key, title, direction) in zip(axes, panels):
        old_values = [old[key] for _, old, _ in usable]
        new_values = [new[key] for _, _, new in usable]
        old_bars = axis.bar([p - width / 2 for p in positions], old_values, width=width,
                            color=SERIES["secondary"], label="TinyBERT-4 INT8 (v1)")
        new_bars = axis.bar([p + width / 2 for p in positions], new_values, width=width,
                            color=SERIES["primary"], label="DeBERTa-v3-xsmall INT8 (v2)")
        for group, values in ((old_bars, old_values), (new_bars, new_values)):
            for bar, value in zip(group, values):
                axis.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=8.5, color=INK)
        axis.set_xticks(list(positions), names)
        axis.set_ylim(0, 1.05)
        axis.set_title(f"{title}\n({direction})", fontsize=11)
        tidy(axis)
    axes[0].set_ylabel("Score")
    axes[0].legend(loc="upper left")

    figure.suptitle(
        "Out-of-distribution behaviour on identical 360-case fixtures, model-only paths",
        fontsize=13, y=1.03,
    )
    save(figure, "ood_model_comparison",
         "TinyBERT INT8 against DeBERTa INT8 on the OOD development and frozen "
         "final fixtures, model-only in both cases.")

    for row in ood_rows:
        print(f"{row['fixture']:20} {row['model']:24} typed F1 {row['typed_f1']:.4f}  "
              f"char recall {row['character_recall']:.4f}  hard-neg FP {row['hard_negative_fp']:.4f}")
else:
    print("skipped: OOD results are not available for both models")

save_table(
    ood_rows,
    ["fixture", "model", "rows", "typed_f1", "typed_precision", "typed_recall",
     "character_recall", "character_leakage", "overmasking", "hard_negative_fp"],
    "ood_model_comparison",
    "Both deployment models on the OOD development and frozen final fixtures.",
)

## 7. Deployment size

What has to fit on the device. Sizes are read from the artifacts themselves
where they exist, and from the recorded metrics where the file is a local build
artifact rather than a committed one.

In [ ]:
size_rows = []


def record_size(model: str, precision: str, path: Path | None, fallback_bytes=None, note: str = ""):
    if path is not None and Path(path).is_file():
        size = Path(path).stat().st_size
        source = Path(path).relative_to(REPO_ROOT).as_posix()
    elif fallback_bytes:
        size, source = int(fallback_bytes), "recorded in saved metrics"
    else:
        return
    size_rows.append({
        "model": model, "precision": precision, "bytes": size,
        "mib": size / 1024 ** 2, "source": source, "note": note,
    })


record_size("DeBERTa-v3-xsmall", "FP32 ONNX",
            MODEL_V2 / "artifacts" / "model_v2" / "models" / "fp32" / "model_fp32.onnx",
            (ACCEPTED or {}).get("model", {}).get("fp32_bytes"),
            "reference graph, not deployed")
record_size("DeBERTa-v3-xsmall", "INT8 ONNX",
            REPO_ROOT / "FullPipeline_V3" / "models" / "deberta_pii" / "model_int8.onnx",
            (ACCEPTED or {}).get("model", {}).get("int8_bytes"),
            "deployed")
if TINYBERT_IID:
    record_size("TinyBERT-4", "FP32 ONNX", None,
                TINYBERT_IID["onnx_fp32"]["model_size_mib"] * 1024 ** 2, "previous generation")
    record_size("TinyBERT-4", "INT8 ONNX",
                TINYBERT_V1 / "onnx_model" / "model_int8.onnx",
                TINYBERT_IID["onnx_int8"]["model_size_mib"] * 1024 ** 2,
                "previous deployment")
record_size("BiLSTM baseline", "FP32 PyTorch",
            REPO_ROOT / "model" / "artifacts" / "pii_masking_bilstm.pt",
            None, "first baseline")

if size_rows:
    ordered = sorted(size_rows, key=lambda row: row["mib"])
    labels = [f"{row['model']}\n{row['precision']}" for row in ordered]
    values = [row["mib"] for row in ordered]
    # One highlight, not a palette: the row labels already carry identity, so
    # colour is spent on the single thing the reader is looking for.
    colours = [
        SERIES["primary"] if row["note"] == "deployed" else "#b8b7b2" for row in ordered
    ]
    figure, axis = plt.subplots(figsize=(9.0, 4.4))
    bars = axis.barh(range(len(labels)), values, height=0.6, color=colours)
    for bar, row in zip(bars, ordered):
        axis.annotate(f"{row['mib']:.1f} MiB" + ("   deployed" if row["note"] == "deployed" else ""),
                      (row["mib"], bar.get_y() + bar.get_height() / 2),
                      va="center", ha="left", xytext=(6, 0), textcoords="offset points",
                      fontsize=9.5, color=INK if row["note"] == "deployed" else MUTED)
    axis.set_yticks(range(len(labels)), labels)
    axis.set_xlim(0, max(values) * 1.32)
    axis.set_xlabel("Artifact size on disk (MiB)")
    axis.set_title("What has to ship to the device", loc="left")
    tidy(axis, x_grid=True)
    save(figure, "model_size_comparison",
         "On-disk size of every privacy-model artifact, deployed one highlighted.")

    for row in ordered:
        print(f"{row['model']:20} {row['precision']:14} {row['mib']:>8.2f} MiB   {row['note']}")
else:
    print("skipped: no artifact sizes available")

save_table(size_rows, ["model", "precision", "bytes", "mib", "source", "note"],
           "model_size_comparison", "On-disk size of each privacy-model artifact.")

## 8. Reference latency by input length

From the workstation preflight benchmark, single-threaded and controlled. It
measures the *architecture* at four input lengths; the head was untrained, so
these numbers say nothing about quality and everything about cost.

This is a development-machine measurement. Raspberry Pi latency comes from
section 9.

In [ ]:
latency_rows = []

if PREFLIGHT and "one_thread_controlled" in PREFLIGHT:
    controlled = PREFLIGHT["one_thread_controlled"]
    models = controlled["models"]
    display = {
        "deberta_fp32": ("DeBERTa-v3-xsmall FP32", SERIES["tertiary"], "--"),
        "deberta_int8": ("DeBERTa-v3-xsmall INT8", SERIES["primary"], "-"),
        "tinybert_int8": ("TinyBERT-4 INT8", SERIES["secondary"], "-"),
    }
    lengths = sorted({int(key) for entry in models.values() for key in entry})

    figure, axis = plt.subplots(figsize=(8.6, 4.8))
    for key, (name, colour, style) in display.items():
        if key not in models:
            continue
        values = [models[key][str(length)]["median_ms"] for length in lengths
                  if str(length) in models[key]]
        present = [length for length in lengths if str(length) in models[key]]
        axis.plot(present, values, marker="o", color=colour, linestyle=style, label=name)
        axis.annotate(f"{values[-1]:.0f} ms", (present[-1], values[-1]),
                      xytext=(6, 0), textcoords="offset points",
                      va="center", fontsize=9.5, color=colour)
        for length in present:
            entry = models[key][str(length)]
            latency_rows.append({
                "model": name, "tokens": length, "repetitions": entry["repetitions"],
                "mean_ms": entry["mean_ms"], "median_ms": entry["median_ms"],
                "p95_ms": entry["p95_ms"], "min_ms": entry["min_ms"], "max_ms": entry["max_ms"],
                "threads": controlled["intra_op_num_threads"],
                "host": PREFLIGHT["host"]["processor"],
            })

    axis.set_xticks(lengths)
    axis.set_xlabel("Input length (tokens)")
    axis.set_ylabel("Median single-inference latency (ms)")
    axis.set_xlim(min(lengths) - 20, max(lengths) + 90)
    axis.set_ylim(0, None)
    axis.set_title(
        "Single-threaded inference cost against input length\n"
        f"{PREFLIGHT['host']['processor']}, {controlled['repetitions']} repetitions "
        f"after {controlled['warmups']} warm-ups",
        loc="left",
    )
    axis.legend(loc="upper left")
    tidy(axis)
    save(figure, "latency_by_input_length",
         "Median single-threaded inference latency against input length for all "
         "three deployment candidates, on the development workstation.")
else:
    print("skipped: no preflight latency benchmark available")

save_table(
    latency_rows,
    ["model", "tokens", "repetitions", "mean_ms", "median_ms", "p95_ms",
     "min_ms", "max_ms", "threads", "host"],
    "latency_by_input_length",
    "Single-threaded latency at four input lengths on the development workstation.",
)

## 9. CPU deployment benchmarks

Whatever `Full_Pipeline_CPU_Benchmark.py` has produced: the 2,000-row text
benchmark. One run gives the latency, stage-breakdown and memory figures. Two
runs on different devices add the comparison and the prediction-parity check.
The end-to-end audio benchmark is section 10.

Nothing here fails when the results do not exist yet.

In [ ]:
RUNS = []
for directory in benchmark_runs:
    config = load_json(directory / "benchmark_config.json", f"{directory.name} config")
    system = load_json(directory / "system_info.json", f"{directory.name} system")
    if not config or not system:
        continue
    RUNS.append({
        "directory": directory,
        "run_id": directory.name,
        "config": config,
        "system": system,
        "latency": load_json(directory / "deberta_latency_summary.json", f"{directory.name} latency"),
        "pipeline": load_json(directory / "pipeline_latency_summary.json", f"{directory.name} pipeline"),
        "evaluation": load_json(directory / "evaluation_metrics.json", f"{directory.name} evaluation"),
        "memory": load_json(directory / "memory_metrics.json", f"{directory.name} memory"),
        "artifacts": load_json(directory / "artifact_manifest.json", f"{directory.name} artifacts"),
    })


def device_name(run) -> str:
    system = run["system"]
    board = system.get("board_model")
    if board:
        return board.replace("Raspberry Pi", "Pi").strip()
    label = run["config"].get("device_label")
    if label:
        return label
    return f"{system.get('os', '?')} {system.get('architecture', '?')}"


if not RUNS:
    print("No CPU benchmark runs found yet.")
    print()
    print("Deployment figures become available once Full_Pipeline_CPU_Benchmark.py")
    print("has been run and its output folder committed. Run it here for a workstation")
    print("baseline, and on the Raspberry Pi for the device comparison, then rerun this")
    print("notebook - nothing needs editing.")
else:
    print(f"{'run':46} {'device':26} {'mode':6} {'rows':>6}")
    print("-" * 88)
    for run in RUNS:
        print(f"{run['run_id']:46} {device_name(run):26} "
              f"{run['config']['mode']:6} {run['config']['rows_scored']:>6,}")

# A smoke run scores ~50 rows to prove the machine works, not to measure it.
# It stays in the run table below - it is evidence that the device ran the
# benchmark - but it must not share an axis with a 2,000-row run, so the
# figures draw from BENCHMARK_RUNS instead.
BENCHMARK_RUNS = [run for run in RUNS if run["config"]["mode"] != "smoke"]
for run in RUNS:
    if run["config"]["mode"] == "smoke":
        UNAVAILABLE.append(
            f"{run['run_id']}: smoke mode ({run['config']['rows_scored']} rows), "
            "listed in the run table but excluded from the comparison figures"
        )

run_rows = [
    {
        "run_id": run["run_id"],
        "device": device_name(run),
        "os": run["system"].get("os"),
        "architecture": run["system"].get("architecture"),
        "cpu": run["system"].get("cpu_model"),
        "logical_cpus": run["system"].get("logical_cpus"),
        "total_ram_gib": (run["system"].get("total_ram_bytes") or 0) / 1024 ** 3,
        "mode": run["config"]["mode"],
        "rows_scored": run["config"]["rows_scored"],
        "int8_sha256": (run["artifacts"] or {}).get("files", {})
        .get("model_int8.onnx", {}).get("sha256"),
        "typed_f1": (run["evaluation"] or {}).get("production_path", {})
        .get("typed_entity", {}).get("f1"),
        "mean_latency_ms": (run["latency"] or {}).get("end_to_end_masking_call", {}).get("mean_ms"),
        "p95_latency_ms": (run["latency"] or {}).get("end_to_end_masking_call", {}).get("p95_ms"),
        "throughput_rows_per_second": (run["latency"] or {}).get("throughput_rows_per_second"),
        "peak_rss_mib": ((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2,
    }
    for run in RUNS
]
save_table(
    run_rows,
    ["run_id", "device", "os", "architecture", "cpu", "logical_cpus", "total_ram_gib",
     "mode", "rows_scored", "int8_sha256", "typed_f1", "mean_latency_ms",
     "p95_latency_ms", "throughput_rows_per_second", "peak_rss_mib"],
    "cpu_benchmark_runs",
    "Every CPU benchmark run discovered, with its device and headline numbers.",
)

In [ ]:
# --- latency distribution -----------------------------------------------------
def read_latency_csv(directory: Path):
    path = directory / "deberta_latency.csv"
    if not path.is_file():
        return []
    with path.open(encoding="utf-8", newline="") as stream:
        return [
            {"latency_ms": float(row["latency_ms"]), "characters": int(row["characters"])}
            for row in csv.DictReader(stream)
        ]


plotted = [(run, read_latency_csv(run["directory"])) for run in BENCHMARK_RUNS]
plotted = [(run, samples) for run, samples in plotted if samples]

if plotted:
    figure, (left, right) = plt.subplots(1, 2, figsize=(12.5, 4.6))
    colours = [SERIES["primary"], SERIES["secondary"], SERIES["tertiary"]]
    percentile_notes = []

    for index, (run, samples) in enumerate(plotted[:3]):
        values = sorted(sample["latency_ms"] for sample in samples)
        colour = colours[index]
        left.hist(values, bins=40, color=colour, alpha=0.55 if index else 0.8,
                  label=f"{device_name(run)} (n={len(values):,})")
        # Empirical CDF: percentiles are read off it directly, no binning artefact.
        right.plot(values, [rank / len(values) for rank in range(1, len(values) + 1)],
                   color=colour, label=device_name(run))
        summary = (run["latency"] or {}).get("end_to_end_masking_call", {})
        marked = [(fraction, key) for fraction, key in
                  ((0.50, "p50_ms"), (0.95, "p95_ms"), (0.99, "p99_ms")) if key in summary]
        for fraction, key in marked:
            right.plot([summary[key]], [fraction], marker="o", color=colour)
        # P95 and P99 sit in the flat top of the CDF, where two devices' labels
        # land on top of each other. One readout per device instead.
        if marked:
            percentile_notes.append((
                device_name(run),
                "   ".join(f"P{int(fraction * 100)} {summary[key]:,.0f} ms"
                           for fraction, key in marked),
                colour,
            ))

    for order, (name, note, colour) in enumerate(percentile_notes):
        right.text(0.98, 0.42 - order * 0.085, f"{name}    {note}",
                   transform=right.transAxes, ha="right", va="center",
                   fontsize=8.5, color=colour)

    left.set_xlabel("Masking latency per row (ms)")
    left.set_ylabel("Rows")
    left.set_title("Latency distribution")
    left.legend(loc="upper right")
    tidy(left)

    right.set_xlabel("Masking latency per row (ms)")
    right.set_ylabel("Share of rows at or below")
    right.set_ylim(0, 1.02)
    right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    right.set_title("Cumulative distribution, with P50 / P95 / P99")
    if len(plotted) > 1:
        right.legend(loc="lower right")
    tidy(right)

    figure.suptitle("End-to-end privacy masking latency on CPU", fontsize=13, y=1.02)
    save(figure, "cpu_latency_distribution",
         "Per-row masking latency distribution and cumulative distribution for "
         "every benchmarked device.")

    percentile_rows = [
        {"run_id": run["run_id"], "device": device_name(run),
         **{key: value for key, value in
            (run["latency"] or {}).get("end_to_end_masking_call", {}).items()}}
        for run, _ in plotted
    ]
    save_table(
        percentile_rows,
        ["run_id", "device", "count", "mean_ms", "std_ms", "min_ms", "p50_ms",
         "p90_ms", "p95_ms", "p99_ms", "max_ms", "total_ms"],
        "cpu_latency_percentiles",
        "Latency percentiles of the full masking call, per device.",
    )
else:
    print("no per-row latency samples recorded yet")

In [ ]:
# --- stage breakdown ----------------------------------------------------------
if BENCHMARK_RUNS:
    run = BENCHMARK_RUNS[-1]
    stage_values = {}
    latency = run["latency"] or {}
    for name, key in (("Neural decode (ONNX + span consolidation)", "neural_decode"),
                      ("Structured regex layer", "structured"),
                      ("Span merge and mask policy", "span_merge")):
        entry = latency.get("stages", {}).get(key)
        if entry and entry.get("count"):
            stage_values[name] = entry["mean_ms"]

    pipeline = (run["pipeline"] or {}).get("stages", {})
    pipeline_labels = {
        "vad_silero": "Silero VAD (3 s synthetic probe)",
        "stt_canary": "Canary STT (3 s synthetic probe)",
        "transcript_merge": "Transcript merge",
        "wake_word_normalisation": "Wake-word normalisation",
        "inactivity_buffer": "Inactivity buffer",
        "privacy_masking": "Privacy masking",
        "digit_normalisation": "Spoken-digit rendering",
    }
    pipeline_values, pipeline_missing = {}, []
    for key, label in pipeline_labels.items():
        entry = pipeline.get(key)
        if not entry:
            continue
        if entry.get("available") and entry.get("count"):
            pipeline_values[label] = entry["mean_ms"]
        else:
            pipeline_missing.append(label)

    if stage_values or pipeline_values:
        panels = int(bool(stage_values)) + int(bool(pipeline_values))
        figure, axes = plt.subplots(panels, 1, figsize=(9.6, 3.0 * panels + 1.0))
        axes = [axes] if panels == 1 else list(axes)

        for axis, (title, values) in zip(
            axes,
            [item for item in (("Inside one masking call", stage_values),
                               ("Across the assembled pipeline", pipeline_values)) if item[1]],
        ):
            names = list(values)
            magnitudes = [values[name] for name in names]
            bars = axis.barh(range(len(names)), magnitudes, height=0.6,
                             color=SERIES["primary"])
            for bar, value in zip(bars, magnitudes):
                axis.annotate(f"{value:.2f} ms", (value, bar.get_y() + bar.get_height() / 2),
                              va="center", ha="left", xytext=(6, 0),
                              textcoords="offset points", fontsize=9.5, color=INK)
            axis.set_yticks(range(len(names)), names)
            # Linear only. A log axis would make the sub-millisecond text stages
            # look comparable to the neural decode, which is the opposite of the
            # point; the annotations carry the small values instead.
            axis.set_xlim(0, max(magnitudes) * 1.30)
            axis.set_xlabel("Mean latency (ms)")
            axis.set_title(title, loc="left")
            axis.invert_yaxis()
            tidy(axis, x_grid=True)

        subtitle = f"{device_name(run)} - {run['config']['mode']} run"
        if pipeline_missing:
            subtitle += f"\nnot measurable on this device: {', '.join(pipeline_missing)}"
        figure.suptitle(f"Where the time goes\n{subtitle}", fontsize=12.5, y=1.0)
        figure.tight_layout()
        save(figure, "pipeline_latency_breakdown",
             "Mean latency of each stage inside a masking call and across the "
             "assembled pipeline, on the most recent benchmarked device. The two "
             "audio stages are driven by a synthesised probe tone rather than "
             "speech; the real-speech cost is in the audio benchmark section.")

        save_table(
            [{"run_id": run["run_id"], "device": device_name(run),
              "scope": scope, "stage": name, "mean_ms": value}
             for scope, values in (("masking_call", stage_values),
                                   ("pipeline", pipeline_values))
             for name, value in values.items()],
            ["run_id", "device", "scope", "stage", "mean_ms"],
            "pipeline_latency_breakdown",
            "Mean latency per stage for the most recent benchmarked device.",
        )
    else:
        print("no stage latency recorded in the most recent run")
else:
    print("no CPU benchmark runs to break down")

In [ ]:
# --- memory and throughput ----------------------------------------------------
memory_runs = [run for run in BENCHMARK_RUNS
                if (run["memory"] or {}).get("process_peak_rss_bytes")]

if memory_runs:
    figure, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4.3))
    names = [device_name(run) for run in memory_runs]
    positions = range(len(names))

    peak = [(run["memory"]["process_peak_rss_bytes"]) / 1024 ** 2 for run in memory_runs]
    stack = [
        (run["memory"].get("privacy_stack_resident_bytes") or 0) / 1024 ** 2
        for run in memory_runs
    ]
    width = 0.36
    peak_bars = left.bar([p - width / 2 for p in positions], peak, width=width,
                         color=SERIES["primary"], label="Process peak RSS")
    stack_bars = left.bar([p + width / 2 for p in positions], stack, width=width,
                          color=SERIES["tertiary"], label="Privacy stack resident")
    for group, values in ((peak_bars, peak), (stack_bars, stack)):
        for bar, value in zip(group, values):
            left.annotate(f"{value:,.0f}", (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 3),
                          textcoords="offset points", fontsize=9, color=INK)
    for index, run in enumerate(memory_runs):
        total = run["system"].get("total_ram_bytes")
        if total:
            left.annotate(f"{total / 1024 ** 3:.1f} GiB RAM", (index, 0),
                          ha="center", va="top", xytext=(0, -18),
                          textcoords="offset points", fontsize=8.5, color=MUTED)
    left.set_xticks(list(positions), names)
    left.set_ylabel("MiB")
    left.set_title("Memory footprint")
    left.legend(loc="upper left")
    tidy(left)

    throughput = [
        (run["latency"] or {}).get("throughput_rows_per_second") or 0 for run in memory_runs
    ]
    bars = right.bar(list(positions), throughput, width=0.5, color=SERIES["primary"])
    for bar, value in zip(bars, throughput):
        right.annotate(f"{value:.2f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 3),
                       textcoords="offset points", fontsize=9.5, color=INK)
    right.set_xticks(list(positions), names)
    right.set_ylabel("Rows masked per second")
    right.set_title("Throughput")
    tidy(right)

    figure.suptitle("Deployment resource use", fontsize=13, y=1.02)
    save(figure, "cpu_memory_and_throughput",
         "Peak process RSS, resident privacy-stack cost and masking throughput "
         "for every benchmarked device.")
else:
    print("no memory metrics recorded yet")

In [ ]:
# --- cross-device parity ------------------------------------------------------
def read_parity(directory: Path) -> dict[str, str]:
    path = directory / "prediction_parity_data.jsonl"
    if not path.is_file():
        return {}
    records = {}
    with path.open(encoding="utf-8") as stream:
        for line in stream:
            if line.strip():
                record = json.loads(line)
                records[record["row_id"]] = record["production_digest"]
    return records


parity_rows = []
comparable = [run for run in BENCHMARK_RUNS
              if (run["evaluation"] or {}).get("production_path")]

if len(comparable) >= 2:
    reference = comparable[0]
    reference_digests = read_parity(reference["directory"])
    for other in comparable[1:]:
        other_digests = read_parity(other["directory"])
        shared = sorted(set(reference_digests) & set(other_digests))
        agree = sum(1 for row_id in shared if reference_digests[row_id] == other_digests[row_id])
        left_metrics = reference["evaluation"]["production_path"]
        right_metrics = other["evaluation"]["production_path"]
        parity_rows.append({
            "reference_run": reference["run_id"],
            "reference_device": device_name(reference),
            "comparison_run": other["run_id"],
            "comparison_device": device_name(other),
            "shared_rows": len(shared),
            "identical_predictions": agree,
            "agreement": agree / len(shared) if shared else None,
            "reference_typed_f1": left_metrics["typed_entity"]["f1"],
            "comparison_typed_f1": right_metrics["typed_entity"]["f1"],
            "typed_f1_delta": right_metrics["typed_entity"]["f1"] - left_metrics["typed_entity"]["f1"],
            "character_recall_delta": right_metrics["private_character_recall"]
            - left_metrics["private_character_recall"],
            "same_model_hash": (reference["artifacts"] or {}).get("files", {})
            .get("model_int8.onnx", {}).get("sha256")
            == (other["artifacts"] or {}).get("files", {})
            .get("model_int8.onnx", {}).get("sha256"),
        })

    figure, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4.3))
    labels = [f"{row['reference_device']}\nvs {row['comparison_device']}" for row in parity_rows]
    agreement = [row["agreement"] or 0 for row in parity_rows]
    positions = range(len(labels))
    bars = left.bar(list(positions), agreement, width=0.45, color=SERIES["primary"])
    for bar, row in zip(bars, parity_rows):
        left.annotate(f"{row['agreement']:.4%}\n{row['identical_predictions']:,}/{row['shared_rows']:,}",
                      (bar.get_x() + bar.get_width() / 2, row["agreement"]),
                      ha="center", va="bottom", xytext=(0, 4),
                      textcoords="offset points", fontsize=9, color=INK)
    left.set_xticks(list(positions), labels)
    left.set_ylim(0, 1.12)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_ylabel("Rows with identical predicted spans")
    left.set_title("Prediction agreement across devices")
    tidy(left)

    deltas = [row["typed_f1_delta"] for row in parity_rows]
    bars = right.bar(list(positions), deltas, width=0.45,
                     color=[SERIES["primary"] if abs(value) < 1e-9 else SERIES["secondary"]
                            for value in deltas])
    for bar, value in zip(bars, deltas):
        right.annotate(f"{value:+.6f}", (bar.get_x() + bar.get_width() / 2, value),
                       ha="center", va="bottom", xytext=(0, 4),
                       textcoords="offset points", fontsize=9, color=INK)
    right.axhline(0, color=MUTED, linewidth=1.0)
    right.set_xticks(list(positions), labels)
    right.set_ylabel("Typed entity F1 difference")
    right.set_title("Metric difference across devices")
    limit = max(0.001, max(abs(value) for value in deltas) * 1.6)
    right.set_ylim(-limit, limit)
    tidy(right)

    figure.suptitle("Same INT8 model, same rows, different CPUs", fontsize=13, y=1.02)
    save(figure, "deployment_parity",
         "Prediction agreement and typed-F1 difference between benchmarked devices "
         "running the identical INT8 artifact.")

    for row in parity_rows:
        print(f"{row['reference_device']} vs {row['comparison_device']}")
        print(f"  shared rows          {row['shared_rows']:,}")
        print(f"  identical predictions {row['identical_predictions']:,} "
              f"({row['agreement']:.4%})")
        print(f"  typed F1 delta        {row['typed_f1_delta']:+.6f}")
        print(f"  same model hash       {row['same_model_hash']}")
else:
    print(f"parity needs two benchmark runs; {len(comparable)} available.")
    print("Run Full_Pipeline_CPU_Benchmark.py on the Raspberry Pi, commit the")
    print("output folder, pull it here and rerun this notebook.")

save_table(
    parity_rows,
    ["reference_run", "reference_device", "comparison_run", "comparison_device",
     "shared_rows", "identical_predictions", "agreement", "reference_typed_f1",
     "comparison_typed_f1", "typed_f1_delta", "character_recall_delta", "same_model_hash"],
    "deployment_parity",
    "Cross-device prediction agreement and metric deltas.",
)

## 10. Audio pipeline deployment benchmark

Whatever `Full_Pipeline_Audio_Benchmark.py` has produced: committed WAV clips
driven through the real deployment, from Silero VAD and Canary STT to the
DeBERTa INT8 masker and the final sanitized text.

This is a **controlled deployment benchmark on a few dozen clips**, not a
speech-recognition benchmark, and it does not replace the held-out text
evaluation in the sections above. The privacy clips are a **synthetic privacy
speech benchmark** - text-to-speech reading sentences written for this project,
with entirely fictional personal information - and are reported separately from
the LibriSpeech dev-clean clips wherever the distinction matters.

Nothing here fails when the results do not exist yet, and a run recorded on a
machine without the speech recognizer still contributes its control arms,
memory profile and VAD timings.


In [ ]:
# --- discover the audio runs ---------------------------------------------------
AUDIO_RUNS = []
for directory in audio_benchmark_runs:
    config = load_json(directory / "benchmark_config.json", f"{directory.name} config")
    system = load_json(directory / "system_info.json", f"{directory.name} system")
    if not config or not system:
        continue
    AUDIO_RUNS.append({
        "directory": directory,
        "run_id": directory.name,
        "config": config,
        "system": system,
        "latency": load_json(directory / "audio_latency_summary.json", f"{directory.name} audio latency"),
        "stt": load_json(directory / "audio_stt_metrics.json", f"{directory.name} audio STT"),
        "privacy": load_json(directory / "audio_privacy_metrics.json", f"{directory.name} audio privacy"),
        "memory": load_json(directory / "audio_memory_metrics.json", f"{directory.name} audio memory"),
    })


def read_audio_clips(directory: Path) -> list[dict]:
    """Per-clip rows, with the numeric columns parsed and blanks left as None."""

    path = directory / "audio_clip_results.csv"
    if not path.is_file():
        return []
    numeric = {
        "audio_duration_s", "expected_entities", "predicted_entities", "wer",
        "audio_load_latency_ms", "vad_latency_ms", "stt_latency_ms",
        "normalization_latency_ms", "privacy_latency_ms", "postprocessing_latency_ms",
        "end_to_end_latency_ms", "stt_real_time_factor", "pipeline_real_time_factor",
        "vad_speech_frame_ratio", "peak_rss_mb",
    }
    rows = []
    with path.open(encoding="utf-8", newline="") as stream:
        for row in csv.DictReader(stream):
            parsed = dict(row)
            for column in numeric:
                value = row.get(column, "")
                parsed[column] = float(value) if value not in ("", None) else None
            rows.append(parsed)
    return rows


for run in AUDIO_RUNS:
    run["clips"] = read_audio_clips(run["directory"])

# The most recent run that actually reached the recognizer is what the
# single-device figures below are drawn from; a run without it can still supply
# the control arms and the memory profile.
AUDIO_WITH_STT = [run for run in AUDIO_RUNS if run["config"].get("stt_available")]
AUDIO_LATEST = (AUDIO_WITH_STT or AUDIO_RUNS)[-1] if AUDIO_RUNS else None

if not AUDIO_RUNS:
    print("No audio pipeline benchmark runs found yet.")
    print()
    print("Section 10 becomes available once Full_Pipeline_Audio_Benchmark.py has been")
    print("run and its output folder committed. Nothing else in this notebook depends")
    print("on it.")
else:
    print(f"{'run':52} {'device':22} {'clips':>5} {'STT':>4} {'WER':>7} {'RTF':>6}")
    print("-" * 100)
    for run in AUDIO_RUNS:
        stt = run["stt"] or {}
        latency = run["latency"] or {}
        wer = (stt.get("overall") or {}).get("aggregate_wer")
        rtf = ((latency.get("real_time_factor") or {}).get("stt") or {}).get("mean_ms")
        print(f"{run['run_id']:52} {device_name(run):22} "
              f"{run['config'].get('clips_scored', 0):>5} "
              f"{'yes' if run['config'].get('stt_available') else 'no':>4} "
              f"{'-' if wer is None else format(wer, '.4f'):>7} "
              f"{'-' if rtf is None else format(rtf, '.3f'):>6}")

# --- the deployment table the poster needs ------------------------------------
# A run that never reached the recognizer measured no word error rate, no
# real-time factor and no end-to-end privacy. The zeros it records are the
# absence of a measurement, not a result, so it stays in the listing above as
# evidence the device ran the benchmark and out of the deployment table.
deployment_rows = []
for run in AUDIO_RUNS:
    if not run["config"].get("stt_available"):
        reason = (run["stt"] or {}).get("reason") or "reason not recorded"
        UNAVAILABLE.append(
            f"{run['run_id']}: speech recognizer unavailable ({reason[:120]}), "
            "listed in the run table but excluded from the deployment summary"
        )
        continue
    latency = run["latency"] or {}
    stages = latency.get("stages") or {}
    rtf = latency.get("real_time_factor") or {}
    privacy = ((run["privacy"] or {}).get("end_to_end") or {})
    stt = ((run["stt"] or {}).get("overall") or {})
    durations = [clip["audio_duration_s"] for clip in run["clips"]
                 if clip["audio_duration_s"] is not None]
    deployment_rows.append({
        "run_id": run["run_id"],
        "device": device_name(run),
        "architecture": run["system"].get("architecture"),
        "audio_clips": run["config"].get("clips_scored"),
        "mean_audio_duration_s": statistics.fmean(durations) if durations else None,
        "total_audio_seconds": sum(durations) if durations else None,
        "stt_available": run["config"].get("stt_available"),
        "aggregate_wer": stt.get("aggregate_wer"),
        "mean_wer": stt.get("mean_wer"),
        "mean_stt_rtf": (rtf.get("stt") or {}).get("mean_ms"),
        "mean_pipeline_rtf": (rtf.get("pipeline") or {}).get("mean_ms"),
        "p50_end_to_end_ms": (stages.get("end_to_end") or {}).get("p50_ms"),
        "p95_end_to_end_ms": (stages.get("end_to_end") or {}).get("p95_ms"),
        "peak_rss_mib": ((run["memory"] or {}).get("process_peak_rss_bytes") or 0) / 1024 ** 2,
        "entity_recall_after_stt": privacy.get("entity_recall_after_stt"),
        "character_recall_after_stt": privacy.get("character_recall_after_stt"),
        "character_leakage_after_stt": privacy.get("character_leakage_after_stt"),
        "fully_sanitized_clip_rate": privacy.get("fully_sanitized_clip_rate"),
        # Spoken numbers the display renderer rebuilt into numerals after the
        # masker had already released the text. None means the run predates the
        # measurement, which is not the same as zero.
        "identifiers_rebuilt_after_masking": (
            (run["privacy"] or {}).get("postprocessing_amplification") or {}
        ).get("identifiers_reconstructed"),
    })

save_table(
    deployment_rows,
    ["run_id", "device", "architecture", "audio_clips", "mean_audio_duration_s",
     "total_audio_seconds", "stt_available", "aggregate_wer", "mean_wer",
     "mean_stt_rtf", "mean_pipeline_rtf", "p50_end_to_end_ms", "p95_end_to_end_ms",
     "peak_rss_mib", "entity_recall_after_stt", "character_recall_after_stt",
     "character_leakage_after_stt", "fully_sanitized_clip_rate",
     "identifiers_rebuilt_after_masking"],
    "audio_deployment_summary",
    "One row per device: audio benchmarked end to end, with word error rate, "
    "real-time factor, latency, memory and end-to-end privacy.",
)


In [ ]:
# --- which stage dominates the runtime ----------------------------------------
STAGE_LABELS = {
    "audio_load": "Audio load",
    "vad": "Silero VAD",
    "stt": "Canary STT",
    "normalization": "Transcript merge / normalisation",
    "privacy": "DeBERTa INT8 masking",
    "postprocessing": "Spoken-digit rendering",
}

if AUDIO_LATEST and (AUDIO_LATEST["latency"] or {}).get("stages"):
    stages = AUDIO_LATEST["latency"]["stages"]
    shares = AUDIO_LATEST["latency"].get("stage_share_of_end_to_end") or {}
    measured = [(label, stages[key]["mean_ms"], shares.get(key, 0.0))
                for key, label in STAGE_LABELS.items()
                if stages.get(key, {}).get("count")]

    if measured:
        figure, (left, right) = plt.subplots(1, 2, figsize=(12.8, 4.4))
        names = [item[0] for item in measured]
        means = [item[1] for item in measured]
        fractions = [item[2] for item in measured]

        bars = left.barh(range(len(names)), means, height=0.6, color=SERIES["primary"])
        for bar, value in zip(bars, means):
            left.annotate(f"{value:,.1f} ms", (value, bar.get_y() + bar.get_height() / 2),
                          va="center", ha="left", xytext=(6, 0),
                          textcoords="offset points", fontsize=9.5, color=INK)
        left.set_yticks(range(len(names)), names)
        left.set_xlim(0, max(means) * 1.32)
        left.set_xlabel("Mean latency per clip (ms)")
        left.set_title("Mean cost per stage", loc="left")
        left.invert_yaxis()
        tidy(left, x_grid=True)

        bottom = 0.0
        palette = [SERIES["primary"], SERIES["secondary"], SERIES["tertiary"],
                   "#8a63d2", "#c9a227", MUTED]
        for index, (name, fraction) in enumerate(zip(names, fractions)):
            right.bar(0, fraction, bottom=bottom, width=0.5,
                      color=palette[index % len(palette)], label=name)
            if fraction > 0.03:
                right.annotate(f"{fraction:.1%}", (0, bottom + fraction / 2),
                               ha="center", va="center", fontsize=9.5, color="white")
            bottom += fraction
        right.set_xlim(-0.6, 1.9)
        right.set_xticks([])
        right.set_ylim(0, 1.0)
        right.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        right.set_ylabel("Share of measured processing time")
        right.set_title("Where the time goes", loc="left")
        right.legend(loc="center right", fontsize=9)
        tidy(right)

        dominant = max(measured, key=lambda item: item[2])
        figure.suptitle(
            f"Full audio pipeline, stage by stage - {device_name(AUDIO_LATEST)}\n"
            f"{dominant[0]} dominates at {dominant[2]:.0%} of processing time",
            fontsize=12.5, y=1.04)
        save(figure, "audio_pipeline_stage_latency",
             "Mean per-clip latency of every pipeline stage from audio to sanitized "
             "text, and each stage's share of total processing time.")

        save_table(
            [{"run_id": AUDIO_LATEST["run_id"], "device": device_name(AUDIO_LATEST),
              "stage": name, "mean_ms": mean, "share_of_processing": fraction}
             for name, mean, fraction in measured],
            ["run_id", "device", "stage", "mean_ms", "share_of_processing"],
            "audio_pipeline_stage_latency",
            "Per-stage mean latency and share of processing time for the audio pipeline.",
        )
else:
    print("no audio stage latency recorded yet")


In [ ]:
# --- how cost scales with clip length -----------------------------------------
scaling = [run for run in AUDIO_RUNS if run["clips"]]

if scaling:
    run = AUDIO_LATEST
    clips = run["clips"]
    subsets = sorted({clip["subset"] for clip in clips})
    colours = {name: colour for name, colour in
               zip(subsets, [SERIES["primary"], SERIES["secondary"], SERIES["tertiary"]])}
    has_stt = any(clip["stt_latency_ms"] is not None for clip in clips)

    panels = 3 if has_stt else 1
    figure, axes = plt.subplots(1, panels, figsize=(4.6 * panels + 0.6, 4.3))
    axes = [axes] if panels == 1 else list(axes)

    def scatter(axis, y_key, y_label, title, scale=1.0):
        for subset in subsets:
            points = [
                (clip["audio_duration_s"], clip[y_key] * scale)
                for clip in clips
                if clip["subset"] == subset
                and clip[y_key] is not None and clip["audio_duration_s"] is not None
            ]
            if points:
                axis.scatter([p[0] for p in points], [p[1] for p in points],
                             color=colours[subset], alpha=0.85, s=42,
                             label=subset.replace("_", " "))
        axis.set_xlabel("Audio duration (s)")
        axis.set_ylabel(y_label)
        axis.set_title(title, loc="left")
        tidy(axis)

    if has_stt:
        scatter(axes[0], "end_to_end_latency_ms", "End-to-end latency (s)",
                "Duration against end-to-end latency", scale=1 / 1000.0)
        # A pipeline that keeps up with speech sits below this line.
        limit = max(clip["audio_duration_s"] for clip in clips)
        axes[0].plot([0, limit], [0, limit], color=MUTED, linestyle="--",
                     linewidth=1.2, label="real time")
        axes[0].legend(loc="upper left", fontsize=9)

        scatter(axes[1], "stt_latency_ms", "STT latency (s)",
                "Duration against STT latency", scale=1 / 1000.0)

        scatter(axes[2], "stt_real_time_factor", "STT real-time factor",
                "Duration against STT RTF")
        axes[2].axhline(1.0, color=SERIES["secondary"], linestyle="--", linewidth=1.3)
        axes[2].annotate("RTF = 1 (real time)", (0.02, 1.0), xycoords=("axes fraction", "data"),
                         xytext=(0, 5), textcoords="offset points",
                         fontsize=9, color=SERIES["secondary"])
        axes[2].set_ylim(bottom=0)
    else:
        scatter(axes[0], "end_to_end_latency_ms", "End-to-end latency (s)",
                "Duration against end-to-end latency (no STT)", scale=1 / 1000.0)

    figure.suptitle(f"Deployment scaling with clip length - {device_name(run)}",
                    fontsize=13, y=1.03)
    figure.tight_layout()
    save(figure, "audio_duration_scaling",
         "Audio duration against end-to-end latency, STT latency and STT real-time "
         "factor. RTF below 1 means the device processes speech faster than it arrives.")

    save_table(
        [{"run_id": run["run_id"], "device": device_name(run), "clip_id": clip["clip_id"],
          "subset": clip["subset"], "duration_bucket": clip["duration_bucket"],
          "audio_duration_s": clip["audio_duration_s"],
          "stt_latency_ms": clip["stt_latency_ms"],
          "end_to_end_latency_ms": clip["end_to_end_latency_ms"],
          "stt_real_time_factor": clip["stt_real_time_factor"],
          "pipeline_real_time_factor": clip["pipeline_real_time_factor"],
          "wer": clip["wer"]}
         for clip in clips],
        ["run_id", "device", "clip_id", "subset", "duration_bucket", "audio_duration_s",
         "stt_latency_ms", "end_to_end_latency_ms", "stt_real_time_factor",
         "pipeline_real_time_factor", "wer"],
        "audio_clip_scaling",
        "Per-clip duration, latency and real-time factor behind the scaling figure.",
    )

    # --- duration bucket against latency --------------------------------------
    buckets = (run["latency"] or {}).get("end_to_end_by_duration_bucket") or {}
    populated = {name: entry for name, entry in buckets.items() if entry.get("count")}
    if len(populated) > 1:
        figure, axis = plt.subplots(figsize=(8.2, 4.2))
        def bucket_start(name: str) -> float:
            """Bucket keys are strings; the axis is ordinal, so sort numerically."""

            try:
                return float(name.split("-", 1)[0].rstrip("s"))
            except ValueError:
                return math.inf

        names = sorted(populated, key=bucket_start)
        positions = range(len(names))
        means = [populated[name]["mean_ms"] / 1000.0 for name in names]
        errors = [
            [(populated[name]["mean_ms"] - populated[name]["min_ms"]) / 1000.0 for name in names],
            [(populated[name]["max_ms"] - populated[name]["mean_ms"]) / 1000.0 for name in names],
        ]
        bars = axis.bar(list(positions), means, width=0.5, color=SERIES["primary"],
                        yerr=errors, capsize=5, ecolor=MUTED)
        for bar, name, value in zip(bars, names, means):
            axis.annotate(f"{value:.2f} s\nn={populated[name]['count']}",
                          (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 16),
                          textcoords="offset points", fontsize=9, color=INK)
        axis.set_xticks(list(positions), names)
        axis.set_ylabel("End-to-end latency (s)")
        axis.set_xlabel("Clip duration bucket")
        axis.set_title(f"End-to-end latency by clip length - {device_name(run)}")
        # Headroom for the two-line annotation that sits above the whisker.
        axis.set_ylim(top=max(populated[name]["max_ms"] for name in names) / 1000.0 * 1.22)
        tidy(axis)
        save(figure, "audio_latency_by_duration_bucket",
             "Mean end-to-end latency per clip-duration bucket, with the observed "
             "minimum and maximum as whiskers.")

        save_table(
            [{"run_id": run["run_id"], "device": device_name(run), "bucket": name,
              **populated[name]} for name in names],
            ["run_id", "device", "bucket", "count", "mean_ms", "std_ms", "min_ms",
             "p50_ms", "p90_ms", "p95_ms", "p99_ms", "max_ms", "total_ms"],
            "audio_latency_by_duration_bucket",
            "End-to-end latency statistics per clip-duration bucket.",
        )
else:
    print("no per-clip audio results recorded yet")


In [ ]:
# --- latency distribution and the memory the device has to find ---------------
if AUDIO_LATEST and AUDIO_LATEST["clips"]:
    run = AUDIO_LATEST
    values = sorted(clip["end_to_end_latency_ms"] / 1000.0 for clip in run["clips"]
                    if clip["end_to_end_latency_ms"] is not None)
    summary = ((run["latency"] or {}).get("stages") or {}).get("end_to_end") or {}
    memory = run["memory"] or {}
    stage_rss = memory.get("stage_rss_bytes") or {}
    stage_order = [("baseline", "Baseline process"), ("vad_loaded", "VAD loaded"),
                   ("stt_loaded", "STT loaded"), ("privacy_loaded", "DeBERTa loaded"),
                   ("all_models_loaded", "All models loaded")]
    stage_points = [(label, stage_rss[key] / 1024 ** 2)
                    for key, label in stage_order if stage_rss.get(key)]
    if memory.get("process_peak_rss_bytes"):
        stage_points.append(("Peak during run",
                             memory["process_peak_rss_bytes"] / 1024 ** 2))

    panels = 1 + int(bool(stage_points))
    figure, axes = plt.subplots(1, panels, figsize=(6.2 * panels, 4.4))
    axes = [axes] if panels == 1 else list(axes)

    left = axes[0]
    if values:
        left.plot(values, [rank / len(values) for rank in range(1, len(values) + 1)],
                  color=SERIES["primary"], marker="o", markersize=4)
        for fraction, key in ((0.50, "p50_ms"), (0.90, "p90_ms"),
                              (0.95, "p95_ms"), (0.99, "p99_ms")):
            if key in summary:
                seconds = summary[key] / 1000.0
                left.plot([seconds], [fraction], marker="o", color=SERIES["secondary"])
                left.annotate(f"P{int(fraction * 100)} {seconds:.2f} s",
                              (seconds, fraction), xytext=(6, -3),
                              textcoords="offset points", fontsize=9,
                              color=SERIES["secondary"])
    left.set_xlabel("End-to-end latency per clip (s)")
    left.set_ylabel("Share of clips at or below")
    left.set_ylim(0, 1.03)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_title(f"End-to-end latency distribution (n={len(values)})", loc="left")
    tidy(left)

    if stage_points:
        right = axes[1]
        names = [item[0] for item in stage_points]
        magnitudes = [item[1] for item in stage_points]
        colours = [SERIES["primary"]] * (len(names) - 1) + [SERIES["secondary"]]
        bars = right.bar(range(len(names)), magnitudes, width=0.55, color=colours)
        for bar, value in zip(bars, magnitudes):
            right.annotate(f"{value:,.0f}", (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=9, color=INK)
        total = run["system"].get("total_ram_bytes")
        # The deployment target is a 4 GB Pi; that budget is drawn, not assumed.
        right.axhline(4096, color=MUTED, linestyle="--", linewidth=1.2)
        right.annotate("4 GiB Raspberry Pi budget", (0.02, 4096),
                       xycoords=("axes fraction", "data"), xytext=(0, 4),
                       textcoords="offset points", fontsize=9, color=MUTED)
        right.set_xticks(range(len(names)), names, rotation=20, ha="right")
        right.set_ylabel("Resident set size (MiB)")
        right.set_ylim(0, max(4096 * 1.15, max(magnitudes) * 1.25))
        right.set_title(f"Memory as the stack loads"
                        + (f" - host has {total / 1024 ** 3:.1f} GiB" if total else ""),
                        loc="left")
        tidy(right)

        save_table(
            [{"run_id": run["run_id"], "device": device_name(run),
              "stage": name, "rss_mib": value} for name, value in stage_points],
            ["run_id", "device", "stage", "rss_mib"],
            "audio_memory_breakdown",
            "Resident memory after each model loads, and the peak during the run.",
        )

    figure.suptitle(f"Audio pipeline cost on {device_name(run)}", fontsize=13, y=1.02)
    figure.tight_layout()
    save(figure, "audio_latency_and_memory",
         "Cumulative distribution of end-to-end latency with P50/P90/P95/P99, and "
         "resident memory as each model loads, against the 4 GiB Raspberry Pi budget.")
else:
    print("no audio latency or memory metrics recorded yet")


In [ ]:
# --- privacy: reference text against the real audio pipeline -------------------
privacy_rows = []
if AUDIO_LATEST and (AUDIO_LATEST["privacy"] or {}).get("control_arms"):
    run = AUDIO_LATEST
    privacy = run["privacy"]
    arms = privacy["control_arms"]
    end = privacy.get("end_to_end") or {}
    stt_ran = bool(run["config"].get("stt_available")) and end.get("expected_entities")

    ARM_LABELS = {
        "reference_text": "Reference text\n(as written)",
        "reference_uncased": "Reference text\n(uncased, unpunctuated)",
    }
    # The "comparable" block scores each gold entity over its own characters with
    # whitespace excluded - the same definition the end-to-end arm has to use,
    # because recognized text carries no gold offsets. Plotting evaluate_spans'
    # character recall beside it would put two denominators on one axis.
    labels, recall, leakage = [], [], []
    for key, label in ARM_LABELS.items():
        if key not in arms:
            continue
        comparable = arms[key].get("comparable")
        if not comparable:
            UNAVAILABLE.append(
                f"{run['run_id']}: control arm {key} predates the comparable-metric "
                "fix; rerun Full_Pipeline_Audio_Benchmark.py"
            )
            continue
        labels.append(label)
        recall.append(comparable["character_recall"])
        leakage.append(comparable["character_leakage"])
        privacy_rows.append({
            "run_id": run["run_id"], "device": device_name(run), "arm": key,
            "typed_entity_f1": arms[key]["typed_entity"]["f1"],
            "character_recall": comparable["character_recall"],
            "character_leakage": comparable["character_leakage"],
            "entity_recall": comparable["entity_recall"],
        })
    if stt_ran:
        labels.append("Audio -> STT -> privacy\n(end to end)")
        recall.append(end["character_recall_after_stt"])
        leakage.append(end["character_leakage_after_stt"])
        privacy_rows.append({
            "run_id": run["run_id"], "device": device_name(run), "arm": "audio_stt_pipeline",
            "typed_entity_f1": None,
            "character_recall": end["character_recall_after_stt"],
            "character_leakage": end["character_leakage_after_stt"],
            "entity_recall": end["entity_recall_after_stt"],
        })

    attribution = privacy.get("failure_attribution") or {}
    panels = 1 + int(bool(stt_ran and sum(
        attribution.get(key, 0) for key in
        ("protected", "privacy_model_caused", "stt_caused", "stt_caused_but_still_masked")
    )))
    figure, axes = plt.subplots(1, panels, figsize=(6.4 * panels, 4.5))
    axes = [axes] if panels == 1 else list(axes)

    left = axes[0]
    positions = range(len(labels))
    width = 0.36
    recall_bars = left.bar([p - width / 2 for p in positions], recall, width=width,
                           color=SERIES["primary"], label="Private character recall")
    leak_bars = left.bar([p + width / 2 for p in positions], leakage, width=width,
                         color=SERIES["secondary"], label="Leakage")
    for group, values in ((recall_bars, recall), (leak_bars, leakage)):
        for bar, value in zip(group, values):
            left.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                          ha="center", va="bottom", xytext=(0, 3),
                          textcoords="offset points", fontsize=9, color=INK)
    left.set_xticks(list(positions), labels, fontsize=9)
    left.set_ylim(0, 1.12)
    left.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
    left.set_ylabel("Private characters")
    left.set_title("Same clips, three routes to the privacy model", loc="left")
    left.legend(loc="upper right", fontsize=9)
    tidy(left)

    if panels == 2:
        right = axes[1]
        buckets = [
            ("Protected", attribution.get("protected", 0), SERIES["primary"]),
            ("STT altered,\nmasked anyway",
             attribution.get("stt_caused_but_still_masked", 0), SERIES["tertiary"]),
            ("Privacy-model\nfailure", attribution.get("privacy_model_caused", 0),
             SERIES["secondary"]),
            ("STT-caused,\nnothing masked", attribution.get("stt_caused", 0), "#8a63d2"),
        ]
        names = [item[0] for item in buckets]
        counts = [item[1] for item in buckets]
        bars = right.bar(range(len(names)), counts, width=0.55,
                         color=[item[2] for item in buckets])
        total = sum(counts) or 1
        for bar, value in zip(bars, counts):
            right.annotate(f"{value}\n{value / total:.0%}",
                           (bar.get_x() + bar.get_width() / 2, value),
                           ha="center", va="bottom", xytext=(0, 3),
                           textcoords="offset points", fontsize=9, color=INK)
        right.set_xticks(range(len(names)), names, fontsize=9)
        right.set_ylabel("Expected private entities")
        right.set_ylim(0, max(counts) * 1.3 if counts else 1)
        right.set_title("Where end-to-end protection succeeds and fails", loc="left")
        tidy(right)

        save_table(
            [{"run_id": run["run_id"], "device": device_name(run),
              "outcome": name.replace("\n", " "), "entities": value,
              "share": value / total} for name, value, _ in buckets],
            ["run_id", "device", "outcome", "entities", "share"],
            "audio_failure_attribution",
            "Every expected private entity, attributed to protection, a "
            "privacy-model failure, or the speech recognizer.",
        )

    # The privacy-annotated clips, not every clip in the run: the LibriSpeech
    # subset is scored for recognition only, so naming its clips here would
    # claim a privacy denominator that does not exist.
    scored_clips = len(privacy.get("privacy_scored_clips") or []) or privacy.get(
        "clips_scored", 0)
    figure.suptitle(
        "Privacy after speech recognition - synthetic privacy speech benchmark\n"
        f"{device_name(run)}, {scored_clips} privacy-annotated clips: a controlled "
        "deployment check, not the model's validation set",
        fontsize=12, y=1.05)
    figure.tight_layout()
    save(figure, "audio_privacy_reference_vs_pipeline",
         "Private-character recall and leakage for the same clips scored from written "
         "reference text, from uncased unpunctuated reference text, and end to end "
         "from audio; plus the attribution of every end-to-end failure.")

    save_table(
        privacy_rows,
        ["run_id", "device", "arm", "typed_entity_f1", "entity_recall",
         "character_recall", "character_leakage"],
        "audio_privacy_arms",
        "Privacy metrics for each route to the model: written reference text, "
        "uncased unpunctuated reference text, and the real audio pipeline.",
    )

    # --- word error rate by subset --------------------------------------------
    by_subset = ((run["stt"] or {}).get("by_subset") or {})
    populated = {name: entry for name, entry in by_subset.items() if entry.get("clips")}
    if len(populated) > 1:
        figure, axis = plt.subplots(figsize=(7.6, 4.2))
        names = list(populated)
        positions = range(len(names))
        aggregate = [populated[name]["aggregate_wer"] for name in names]
        median = [populated[name]["median_wer"] for name in names]
        width = 0.36
        left_bars = axis.bar([p - width / 2 for p in positions], aggregate, width=width,
                             color=SERIES["primary"], label="Aggregate WER")
        right_bars = axis.bar([p + width / 2 for p in positions], median, width=width,
                              color=SERIES["tertiary"], label="Median clip WER")
        for group, values in ((left_bars, aggregate), (right_bars, median)):
            for bar, value in zip(group, values):
                axis.annotate(f"{value:.3f}", (bar.get_x() + bar.get_width() / 2, value),
                              ha="center", va="bottom", xytext=(0, 3),
                              textcoords="offset points", fontsize=9, color=INK)
        axis.set_xticks(list(positions),
                        [f"{name.replace('_', ' ')}\n({populated[name]['clips']} clips)"
                         for name in names])
        axis.yaxis.set_major_formatter(PercentFormatter(xmax=1.0))
        axis.set_ylabel("Word error rate")
        axis.set_title(f"Speech recognition on each subset - {device_name(run)}")
        axis.legend(loc="upper right", fontsize=9)
        tidy(axis)
        save(figure, "audio_word_error_rate",
             "Word error rate on the synthetic privacy clips and on the LibriSpeech "
             "dev-clean clips, reported separately because they are different "
             "distributions.")

        save_table(
            [{"run_id": run["run_id"], "device": device_name(run), "subset": name,
              **populated[name]} for name in names],
            ["run_id", "device", "subset", "clips", "reference_words", "aggregate_wer",
             "mean_wer", "median_wer", "min_wer", "max_wer", "substitutions",
             "deletions", "insertions"],
            "audio_word_error_rate",
            "Word error rate per fixture subset.",
        )
else:
    print("no audio privacy metrics recorded yet")


In [ ]:
# --- what the render step rebuilt after masking --------------------------------
# Canary runs with pnc=no, so a spoken number reaches the model as digit words.
# Where the model missed one, the display renderer - which runs after masking by
# design - rewrote the surviving words into a numeral. That leak is already
# counted as a privacy-model failure above, so it is reported beside the
# attribution rather than added to it: one entity, counted once.
amplification = ((AUDIO_LATEST or {}).get("privacy") or {}).get(
    "postprocessing_amplification"
)
if AUDIO_LATEST and amplification is None and (AUDIO_LATEST["privacy"] or {}).get("end_to_end"):
    UNAVAILABLE.append(
        f"{AUDIO_LATEST['run_id']}: predates the postprocessing-amplification "
        "measurement; rerun Full_Pipeline_Audio_Benchmark.py"
    )
    print("this run predates the spoken-digit measurement; rerun the audio benchmark")
elif amplification and amplification.get("cases"):
    run = AUDIO_LATEST
    cases = amplification["cases"]
    save_table(
        [
            {
                "run_id": run["run_id"],
                "device": device_name(run),
                "clip_id": case["clip_id"],
                "subset": case["subset"],
                "unmasked_entity_labels": ", ".join(case["unmasked_entity_labels"]),
                "digits": case["digits"],
                "rendered_value": case["rendered_value"],
                "privacy_input_text": case["privacy_input_text"],
                "final_masked_output": case["final_masked_output"],
            }
            for case in cases
        ],
        ["run_id", "device", "clip_id", "subset", "unmasked_entity_labels", "digits",
         "rendered_value", "privacy_input_text", "final_masked_output"],
        "audio_postprocessing_amplification",
        "Spoken numbers the privacy model left unmasked and the display renderer "
        "then rewrote into numerals of identifier length. Every value is fictional: "
        "telephone numbers come from the NANP 555-01xx range reserved for fiction.",
    )

    lines = []
    for case in cases:
        labels = ", ".join(case["unmasked_entity_labels"]) or "no gold entity"
        lines.append(f"{case['clip_id']}  ({labels})")
        lines.append(f"  heard     {case['privacy_input_text']}")
        lines.append(f"  released  {case['final_masked_output']}")
        lines.append("")
    lines = lines[:-1]

    # The canvas is sized from the text block rather than the other way round:
    # a title, one 9 pt monospace line per entry, then the footnote.
    LINE_INCHES = 9 * 1.5 / 72
    figure, axis = plt.subplots(figsize=(9.6, 0.55 + LINE_INCHES * len(lines) + 0.66))
    axis.axis("off")
    axis.set_title(
        f"Spoken numbers rebuilt after masking - "
        f"{amplification['identifiers_reconstructed']} identifier(s) on "
        f"{amplification['clips_affected']} clip(s), {device_name(run)}",
        loc="left", fontsize=12, pad=8,
    )
    axis.text(0.0, 1.0, "\n".join(lines), va="top", ha="left", fontsize=9,
              family="monospace", linespacing=1.5, color=INK,
              transform=axis.transAxes)
    figure.text(
        0.012, 0.02,
        f"Identifier scale is {amplification['identifier_digits']}+ digits, taken from "
        "the deployment's own structured recognizer. The rendering step is display-only "
        "- storage keeps the words - so this is\nthe form the leak takes on screen, not "
        "additional leaked information. Every value is fictional.",
        va="bottom", ha="left", fontsize=8.5, color=MUTED,
    )
    figure.tight_layout(rect=(0, 0.13, 1, 1))
    save(figure, "audio_postprocessing_amplification",
         "The spoken-digit finding: utterances where the privacy model did not mask a "
         "spoken number and the display renderer then rendered it as a numeral.")
    print(f"{amplification['identifiers_reconstructed']} rebuilt identifier(s) on "
          f"{amplification['clips_affected']} clip(s)")
elif amplification:
    print("no spoken number was rebuilt after masking in this run")
else:
    print("no audio privacy metrics recorded yet")


## 10. Index and overview

In [ ]:
figures = [item for item in GENERATED if item["kind"] == "figure"]
tables = [item for item in GENERATED if item["kind"] == "table"]

index = {
    "generated_from": "Research_Paper_Figures.ipynb",
    "repository_root": REPO_ROOT.name,
    "figures": figures,
    "tables": tables,
    "cpu_benchmark_runs": [run["run_id"] for run in RUNS],
    "audio_benchmark_runs": [run["run_id"] for run in AUDIO_RUNS],
    "unavailable_inputs": UNAVAILABLE,
}
index_path = SUMMARIES / "figure_index.json"
with index_path.open("w", encoding="utf-8", newline="\n") as stream:
    stream.write(json.dumps(index, indent=2) + "\n")

lines = [
    "# Research results overview",
    "",
    "Generated by `Research_Paper_Figures.ipynb`. Every number here comes from a "
    "saved evaluation artifact in this repository; nothing was recomputed.",
    "",
    f"- Figures: **{len(figures)}** (PNG at 300 dpi and PDF, in `research_results/figures/`)",
    f"- Tables: **{len(tables)}** (CSV, in `research_results/tables/`)",
    f"- CPU text benchmark runs discovered: **{len(RUNS)}**",
    f"- Audio pipeline benchmark runs discovered: **{len(AUDIO_RUNS)}**",
    "",
]

if split_rows:
    deployed = {row["view"]: row for row in split_rows}
    lines += ["## Deployed model headline", "",
              "| View | Rows | Typed F1 | Character recall | Leakage |", "|---|---:|---:|---:|---:|"]
    for view, row in deployed.items():
        lines.append(
            f"| {view} | {row['rows']:,} | {row['typed_f1']:.4f} | "
            f"{row['private_character_recall']:.4f} | {row['private_character_leakage']:.4f} |"
        )
    lines.append("")

if quantization_rows:
    worst = max(quantization_rows, key=lambda row: row["typed_f1_drop"])
    lines += [
        "## Quantization",
        "",
        f"INT8 costs at most **{worst['typed_f1_drop']:.4f}** typed entity F1 "
        f"(on `{worst['view']}`), against a release gate of {GATE_LIMIT}. "
        "Every view passed.",
        "",
    ]

if ood_rows:
    lines += ["## Out of distribution", "",
              "| Fixture | Model | Typed F1 | Character recall | Hard-negative FP |",
              "|---|---|---:|---:|---:|"]
    for row in ood_rows:
        lines.append(
            f"| {row['fixture']} | {row['model']} | {row['typed_f1']:.4f} | "
            f"{row['character_recall']:.4f} | {row['hard_negative_fp']:.4f} |"
        )
    lines.append("")

if run_rows:
    lines += ["## Deployment benchmarks", "",
              "| Device | Mode | Rows | Typed F1 | Mean latency | Peak RSS |",
              "|---|---|---:|---:|---:|---:|"]
    for row in run_rows:
        lines.append(
            f"| {row['device']} | {row['mode']} | {row['rows_scored']:,} | "
            f"{row['typed_f1']:.4f} | {row['mean_latency_ms']:.1f} ms | "
            f"{row['peak_rss_mib']:,.0f} MiB |"
        )
    lines.append("")
else:
    lines += [
        "## Deployment benchmarks",
        "",
        "None yet. Run `Full_Pipeline_CPU_Benchmark.py` on the workstation and on "
        "the Raspberry Pi, commit the folders it writes, then rerun this notebook.",
        "",
    ]

if deployment_rows:
    lines += [
        "## Audio pipeline deployment benchmark",
        "",
        "A controlled end-to-end benchmark on a few dozen clips - a **synthetic "
        "privacy speech benchmark** plus a LibriSpeech dev-clean subset. Not a "
        "speech-recognition benchmark, and not a replacement for the held-out text "
        "evaluation above.",
        "",
        "| Device | Clips | Mean WER | Mean STT RTF | Mean pipeline RTF | P95 latency | "
        "Peak RSS | Character recall | Leakage |",
        "|---|---:|---:|---:|---:|---:|---:|---:|---:|",
    ]
    for row in deployment_rows:
        def cell(value, spec=".4f"):
            return "n/a" if value is None else format(value, spec)

        lines.append(
            f"| {row['device']} | {row['audio_clips']} | {cell(row['aggregate_wer'])} | "
            f"{cell(row['mean_stt_rtf'], '.3f')} | {cell(row['mean_pipeline_rtf'], '.3f')} | "
            f"{cell(row['p95_end_to_end_ms'], ',.0f')} ms | "
            f"{cell(row['peak_rss_mib'], ',.0f')} MiB | "
            f"{cell(row['character_recall_after_stt'])} | "
            f"{cell(row['character_leakage_after_stt'])} |"
        )
    lines.append("")

lines += ["## Figures", ""]
lines += [f"- `{item['name']}` - {item['caption']}" for item in figures] or ["- none"]
lines += ["", "## Tables", ""]
lines += [f"- `{item['name']}.csv` ({item['rows']} rows) - {item['caption']}"
          for item in tables] or ["- none"]

if UNAVAILABLE:
    lines += ["", "## Inputs not found", ""]
    lines += [f"- {message}" for message in UNAVAILABLE]
lines.append("")

overview_path = SUMMARIES / "results_overview.md"
overview_path.write_text("\n".join(lines), encoding="utf-8", newline="\n")

print(f"{len(figures)} figures, {len(tables)} tables")
print(f"wrote {index_path.relative_to(REPO_ROOT).as_posix()}")
print(f"wrote {overview_path.relative_to(REPO_ROOT).as_posix()}")
if UNAVAILABLE:
    print("\ninputs not found:")
    for message in UNAVAILABLE:
        print(" ", message)

---

## Done

`research_results/figures/` holds a PNG and a PDF of every plot; drop either
straight into the report. `research_results/tables/` holds the numbers behind
them, so a figure can be redrawn or a value quoted without reopening this
notebook. `research_results/summaries/results_overview.md` is the written
digest.

When the Raspberry Pi benchmarks land, pull them and press Run All again.
Sections 9 and 10 pick them up, and the device-comparison and parity figures
appear on their own.